CELL 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from prophet import Prophet

plt.style.use('default')


CELL 2 — Load Dataset

In [ ]:
# Update path if needed
df = pd.read_csv("data/sales_data.csv")
df.head()


CELL 3 — Basic Data Checks




In [ ]:
df.info()
df.isnull().sum()


CELL 4 — Data Cleaning

In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle missing values
df.fillna(method='ffill', inplace=True)

# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])


CELL 5 — Monthly Sales Aggregation

In [ ]:
monthly_sales = (
    df.groupby(pd.Grouper(key='Order Date', freq='M'))['Sales']
    .sum()
    .reset_index()
)

monthly_sales.head()


🟦 CELL 6 — Feature Engineering

In [ ]:
monthly_sales['Month'] = monthly_sales['Order Date'].dt.month
monthly_sales['Year'] = monthly_sales['Order Date'].dt.year
monthly_sales['Rolling_Avg'] = monthly_sales['Sales'].rolling(window=3).mean()

monthly_sales.head()


🟦 CELL 7 — Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly_sales['Order Date'], monthly_sales['Sales'], label='Monthly Sales')
plt.title("Monthly Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.show()


🟦 CELL 8 — Prepare Data for Prophet

In [ ]:
df_prophet = monthly_sales.rename(columns={
    'Order Date': 'ds',
    'Sales': 'y'
})

df_prophet.head()


🟦 CELL 9 — Train Forecasting Model

In [ ]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

model.fit(df_prophet)


🟦 CELL 10 — Generate Future Forecast (12 Months)

In [ ]:
future = model.make_future_dataframe(periods=12, freq='M')
forecast = model.predict(future)

forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].head()


🟦 CELL 11 — Forecast Visualization

In [ ]:
model.plot(forecast)
plt.title("Sales Forecast")
plt.show()


🟦 CELL 12 — Trend & Seasonality Components

In [ ]:
model.plot_components(forecast)
plt.show()


🟦 CELL 13 — Export Forecast for Power BI

In [ ]:
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(
    "outputs/forecast.csv", index=False
)


🟦 CELL 14 — Save Forecast Plot (for GitHub)

In [ ]:
fig = model.plot(forecast)
fig.savefig("outputs/forecast_plot.png")
